# 06 — Putting It Together: The Full SupportPilot Pipeline
### Chains + Tools + Retrieval + deterministic escalation, end to end

This is the capstone notebook of the series. Every concept from notebooks
1-5 shows up here, wired into `pipeline.py`'s `run_ticket()` function. By the
end you should be able to trace one ticket through every layer and explain,
in your own words, why each layer is built the way it is.

## 6.1 Setup

In [ ]:
import database
database.init_db(force=True)
print(f"Database seeded at {database.DB_PATH}")

import os
print(f"SUPPORTPILOT_MODE = {os.environ.get('SUPPORTPILOT_MODE', 'mock')}  (default: mock, no API key needed)")


## 6.2 The pieces, recapped

| Notebook | Concept | Where it lives in this project |
|---|---|---|
| 1 | Messages, chat models, prompts | `chains.py`'s `_get_chat_model()` + `ChatPromptTemplate` calls |
| 2 | Structured output | `chains.py`'s `with_structured_output(TicketClassification)` / `ValidationResult` |
| 3 | LCEL — Runnables, `\|`, `RunnableParallel` | `chains.py`'s chain construction, `pipeline.py`'s `_context_gathering` |
| 4 | Tools | `tools.py`'s `@tool`-decorated DB/KB functions |
| 5 | Embeddings & retrieval | `embeddings.py` + `retrieval.py`'s FAISS-backed `KnowledgeRetriever` |
| *(deliberately not LangChain)* | Escalation rules | `escalation_rules.py` — plain Python, on purpose |

Let's run one ticket through the whole thing and watch each piece fire.

In [ ]:
from pipeline import run_ticket, pretty_print_report

report = run_ticket(
    ticket_id="T-NOTEBOOK-001",
    ticket_text="Where is my order? It's been 6 days and tracking hasn't updated.",
    customer_id="CUST001",
)
pretty_print_report(report)


In [ ]:
# The agent_trace field is exactly the layer-by-layer path notebooks 1-5 covered:
for step in report["agent_trace"]:
    print(f"  -> {step}")


## 6.3 Tracing a hard-escalation case

Now run a fraud-adjacent ticket and watch the trace differ at the very last
step — everything up through validation runs identically, but
`escalation_rules.decide_escalation()` (not a chain, not a tool call) makes
the final call.

In [ ]:
fraud_report = run_ticket(
    ticket_id="T-NOTEBOOK-002",
    ticket_text="There's a transaction on my account that I don't recognize at all.",
    customer_id="CUST005",
)
pretty_print_report(fraud_report)
print()
print("Escalation detail:", fraud_report["escalation"])


## 6.4 Proving the hard rule survives rephrasing

This is the check that matters most for the capstone rubric: the *same*
underlying issue, phrased more softly, must still escalate. If it doesn't,
the escalation logic has a prompt-shaped hole in it.

In [ ]:
softened_report = run_ticket(
    ticket_id="T-NOTEBOOK-003",
    ticket_text="Hey, quick one - I noticed a small charge on my account, just wanted "
                "to double check, I'm not sure it was me.",
    customer_id="CUST005",
)
pretty_print_report(softened_report)

assert fraud_report["resolution_path"] == "escalated"
assert softened_report["resolution_path"] == "escalated"
print("\nBoth phrasings escalated. The hard rule held.")


## 6.5 Running the full test suite from the notebook

`test_tickets.py` has all 8 scenarios from the design doc. Run it here to
see the whole matrix at once.

In [ ]:
from test_tickets import run_suite
run_suite()


## 6.6 What you'd change for `live` mode

Set these two lines *before* importing anything from this project (LangChain
chains are built once, at import time, based on `MODE`) and every chain in
`chains.py` switches from the offline heuristics to real `ChatAnthropic`
calls with structured output — nothing else in `pipeline.py`, `tools.py`, or
`retrieval.py` needs to change:

```python
import os
os.environ["SUPPORTPILOT_MODE"] = "live"
os.environ["ANTHROPIC_API_KEY"] = "sk-..."

# now import/run the pipeline
from pipeline import run_ticket
run_ticket("T001", "Where is my order?", customer_id="CUST001")
```

## 6.7 Where to go from here

You've now covered messages/prompts, structured output, LCEL composition,
tools, and retrieval — the LangChain half of what the course calls a
"pipeline/graph-based" framework. Two natural next steps, both hinted at in
the project's README:

1. **Dynamic orchestration.** Right now `pipeline.py`'s step order is fixed.
   Week 4's `SelectorGroupChat` pattern (AutoGen) or CrewAI's role-based
   orchestration would let an agent *decide*, e.g., whether a ticket needs
   KB retrieval at all — useful for pure account-status questions where
   retrieval is wasted work.
2. **MCP.** `tools.py`'s tools are already framework-native `Tool` objects.
   Exposing them behind an MCP server (Week 5) instead of calling them
   directly from Python is a small step from here, and is exactly what lets
   a completely different AI client reuse the same account/KB access without
   rewriting any of this code.

## Final exercise

Pick one test case from `test_tickets.py`, and without looking at
`pipeline.py`, write out by hand (in a markdown cell) every LangChain
concept you'd expect to fire and in what order, based on notebooks 1-5.
Then run it and compare your prediction to the real `agent_trace`.